# Random Seed

In [1]:
import random
import torch
import numpy as np
import json


In [2]:
print(torch.__version__)

2.3.1+rocm6.0


In [3]:

def set_seed(seed):
    # 設定 Python 隨機數生成器的種子
    random.seed(seed)
    
    # 設定 numpy 隨機數生成器的種子
    np.random.seed(seed)
    
    # 設定 PyTorch 隨機數生成器的種子
    torch.manual_seed(seed)
    
    # 如果使用 GPU，設置 CUDA 隨機數生成器的種子
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 設置 PyTorch 預測模式，保證可重現性
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 設定隨機種子
set_seed(22)


## Dataset

In [4]:
from transformers import AutoTokenizer
import torch

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-pretrain", local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/sec-bert-base")

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)

# 設定預設值
CLASSIFICATION_MISSING_VALUE = -100
NUMERIC_MISSING_VALUE = torch.finfo(torch.float32).max  # 3.4028235e+38

/home/mo1om/code/miniconda3/envs/selfmix/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# IterableDataset

with open('../processed_data_task1_smaller/counter/tag_count_train_400k.json', 'r', encoding = 'utf-8') as file:
# with open('processed_iterable_dataset/counter/train_8k.json', 'r', encoding = 'utf-8') as file:
# with open('../processed_data_task1/counter/tag_count_train_data.json', 'r', encoding = 'utf-8') as file:    
    tag_counter = json.load(file)
    
# 想讓數量多的類別在前面

tag_counter = dict(sorted(tag_counter.items(), key = lambda item:item[1], reverse=True))
print(len(tag_counter))
count_threshold = 10
standard_rare_tags = {tag for tag, count in tag_counter.items() if count < count_threshold}
tag_list = [tag for tag in tag_counter.keys() if tag not in standard_rare_tags]
print(tag_list[:5])
print(f'Length of standard_rare_tags: {len(standard_rare_tags)}')
print(f'Length of all tags: {len(tag_list)}')

id2tag = {idx: tag for idx, tag in enumerate(tag_list)}
tag2id = {tag: idx for idx, tag in enumerate(tag_list)}

time_list = ['instant; past', 'instant; current', 'instant; future', 'period; past', 'period; current', 'period; future', 'period; past_current', 'period; current_future', 'period; past_future']
id2time = {idx: time for idx, time in enumerate(time_list)}
time2id = {time: idx for idx, time in enumerate(time_list)}

scale_list = [str(i) for i in range(-12, 13)]
id2scale = {idx: scale for idx, scale in enumerate(scale_list)}
scale2id = {scale: idx for idx, scale in enumerate(scale_list)}
# with open('processed_data_task1_smaller/counter/train_100k.json')

978
['custom', 'standard_rare', 'us-gaap:DebtInstrumentInterestRateStatedPercentage', 'us-gaap:DebtInstrumentBasisSpreadOnVariableRate1', 'us-gaap:DebtInstrumentFaceAmount']
Length of standard_rare_tags: 0
Length of all tags: 978


NameError: name 'id2signed_scale' is not defined

In [6]:
# %%
# 繼承自現有的 scale_list 和 scale2id
# scale_list: [-12, -11, ..., 11, 12]
# scale2id: {"-12": 0, ..., "0": 12, ..., "12": 24}

# 總共 50 個 Signed Scale 類別
signed_scale_list = []
# 0-24: positive/non-negative
for scale_str in scale_list:
    signed_scale_list.append(f"0_{scale_str}") 
# 25-49: negative
for scale_str in scale_list:
    signed_scale_list.append(f"1_{scale_str}") 

id2signed_scale = {idx: ss for idx, ss in enumerate(signed_scale_list)}
signed_scale2id = {ss: idx for idx, ss in enumerate(signed_scale_list)}

NUM_SIGNED_SCALES = len(signed_scale_list)
print(f"Total Signed Scale Classes: {NUM_SIGNED_SCALES}")  
signed_scale_list

Total Signed Scale Classes: 50


['0_-12',
 '0_-11',
 '0_-10',
 '0_-9',
 '0_-8',
 '0_-7',
 '0_-6',
 '0_-5',
 '0_-4',
 '0_-3',
 '0_-2',
 '0_-1',
 '0_0',
 '0_1',
 '0_2',
 '0_3',
 '0_4',
 '0_5',
 '0_6',
 '0_7',
 '0_8',
 '0_9',
 '0_10',
 '0_11',
 '0_12',
 '1_-12',
 '1_-11',
 '1_-10',
 '1_-9',
 '1_-8',
 '1_-7',
 '1_-6',
 '1_-5',
 '1_-4',
 '1_-3',
 '1_-2',
 '1_-1',
 '1_0',
 '1_1',
 '1_2',
 '1_3',
 '1_4',
 '1_5',
 '1_6',
 '1_7',
 '1_8',
 '1_9',
 '1_10',
 '1_11',
 '1_12']

In [7]:
import locale
from word2number import w2n

def convert_span_to_number(span):
    """
    將 span 轉換為數字。
    """
    span = span.strip()
    
    # 嘗試直接轉換為數字
    try:
        return locale.atof(span.replace(",", ""))  # 去掉千分位逗號並轉換
    except ValueError:
        pass  # 不是標準數字，繼續嘗試解析
    
    # 嘗試將文字轉為數字
    try:
        return w2n.word_to_num(span.lower())
    except ValueError:
        pass  # 不是可解析的數字
    
    return None  # 解析失敗，返回 None


In [8]:
# batch
from concurrent.futures import ThreadPoolExecutor
import math
from tqdm import tqdm

def process_batch(batch, target_attrs, tokenizer):

    batch_results = []
    for job in batch:
        context_p = job['context'].get("context_p", "")
        context_t = job['context'].get("context_t", "")
        context_n = job['context'].get("context_n", "")
        # full_context = f"{context_t} [SEP] {context_p} [SEP] {context_n}"
        document_info = f"{job['document']['document_type']};{job['document']['period_end_date']};{job['document']['fiscal_year']};{job['document']['period_focus']}"
        full_context = f"{context_t} [SEP] {document_info} [SEP] {context_p} [SEP] {context_n}"

        tokenized = tokenizer(
            full_context,
            padding = "max_length",
            truncation = True,
            max_length = 512,
            return_tensors = "pt",
            return_offsets_mapping = True,
            return_special_tokens_mask = True,
        )

        token_ids = tokenized["input_ids"].squeeze(0)
        offset_mapping = tokenized["offset_mapping"].squeeze(0)

        # sep_indices = [idx for idx, token in enumerate(token_ids) if token == tokenizer.sep_token_id]
        # context_t_end = sep_indices[0] if sep_indices else len(token_ids) - 1
        
        for i, target in enumerate(job['targets']):
            start_char, end_char = target['start_pos'], target['end_pos']
            start_token, end_token = -1, -1

            for idx, (start, end) in enumerate(offset_mapping):
                # if idx > context_t_end:
                #     break
                if start <= start_char < end:
                    start_token = idx
                if start < end_char <= end:
                    end_token = idx
                    break
            
            # if start_token == -1 or end_token == -1 or start_token > context_t_end or end_token > context_t_end:
            if start_token == -1 or end_token == -1:
                continue 
                
            # print("\n=== DEBUG: Tokenization ===")
            # token_ids = tokenized["input_ids"].squeeze(0) 
            # tokens = tokenizer.convert_ids_to_tokens(token_ids.tolist())
            # print("Original Text:", full_context)
            # # print("Tokens:", tokens)
            # print("Offset Mapping:", offset_mapping)
            # print(f"Target Text: {target['text']} | Start Char: {start_char}, End Char: {end_char}")
            # print(f"Found Token Indices -> Start: {start_token}, End: {end_token}")
            # if start_token >= 0 and end_token >= 0:
            #     print(f"Matched Tokens: {tokens[start_token:end_token+1]}")
            # print("====================================\n")
            
            target_data = {
                "job_id": job["job_id"],
                "seq_id": target["seq_id"],
                "context": full_context,
                "input_ids": token_ids,
                "attention_mask": tokenized['attention_mask'].squeeze(0),
                "start_token": start_token,
                "end_token": end_token,
                "value": convert_span_to_number(target['text']),
                "doc_link": job['document']['document_link'],
            }
            
            for attr in target_attrs:
                if attr in ["tag", "time", "scale", "negative","signed_scale"]:  # 分類屬性
                    target_data[attr] = CLASSIFICATION_MISSING_VALUE
                elif attr == "fact":  # 數值屬性
                    target_data[attr] = NUMERIC_MISSING_VALUE

            gold_values = job['golds'][i]['value']
            for attr_idx, attr in enumerate(target['attribute']):
                value = gold_values[attr_idx]
                
                if attr == 'tag':
                    # 暫時先歸到 standard_rare 
                    if value in standard_rare_tags:
                        value = 'standard_rare'
                    target_data['tag'] = tag2id.get(value, -100)
                elif attr == 'time':
                    target_data['time'] = time2id.get(value, -100)
                elif attr == 'fact':
                    if value:
                        fact_val = float(value)
                        target_data['fact'] = fact_val
                        
                        # Bug Fix 1: 負號邏輯 (比較 float 而不是 string)
                        is_negative = 1 if fact_val < 0 else 0
                        target_data['negative'] = is_negative
                        
                        derived_scale_id = None
                        
                        # Bug Fix 2: 使用 abs() 避免 log 錯誤
                        # Bug Fix 3: 計算比例 (Fact / Text) 而不是 Fact 的 log
                         
                        try:
                            ratio = abs(fact_val) 
                            log_scale = math.log10(ratio)
                            calculated_scale = int(round(log_scale)) # e.g., 6
                            
                            # 轉成字串來查 ID (假設 scale2id key 是字串 "-12" ~ "12")
                            derived_scale_id = scale2id.get(str(calculated_scale), -100)
                        except ValueError:
                            derived_scale_id = -100
                         

                        # Bug Fix 4 & 5: 使用數學公式計算 Signed ID，並存入 target_data
                        # 假設 ID 0-24 是正數區，25-49 是負數區
                        if derived_scale_id is not None and derived_scale_id != -100:
                            num_scale_classes=len(scale_list)
                            signed_id = derived_scale_id + (num_scale_classes * is_negative)
                            target_data['signed_scale'] = signed_id
                        else:
                            target_data['signed_scale'] = CLASSIFICATION_MISSING_VALUE
                    # target_data['fact'] = float(value)
                elif attr == 'scale':
                    target_data['scale'] = scale2id.get(value, -100)
                   

            # 3. 處理 Signed Scale 
             
            # 確保 negative 和 scale 即使有 fact 缺失也可能被設定（如果它們是獨立標註的）
            # (根據您的原始代碼，如果 fact 存在，negative 已經在 fact 區塊被計算)
            # if current_negative is not None:
            #     target_data['negative'] = current_negative # 如果 fact 不為 None   
                    
            batch_results.append(target_data)
    return batch_results

from concurrent.futures import ProcessPoolExecutor, TimeoutError


def process_batch_wrapper(args):
    """ 用於 `ProcessPoolExecutor` 的批次處理函數 """
    batch, target_attrs, tokenizer = args
    return process_batch(batch, target_attrs, tokenizer)

def process_data(data, target_attrs, tokenizer, batch_size = 32, num_workers = 8):
    inputs = []
    
    # 計算總批次數
    num_batches = math.ceil(len(data) / batch_size)

    # 將數據拆分成批次
    batches = [data[i * batch_size: (i + 1) * batch_size] for i in range(num_batches)]

    # 構建參數列表
    task_args = [(batch, target_attrs, tokenizer) for batch in batches]

    # 使用多進程處理批次
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(tqdm(executor.map(process_batch_wrapper, task_args), total=num_batches, desc="Processing Data"))

    # 合併所有批次的結果
    for res in results:
        inputs.extend(res)
    
    return inputs


#### IterableDataset

### preprocess


#### Load Processed Iterable Dataset

In [9]:
# 讀取處理後的 JSONL

import json
import torch
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

class ProcessedIterableDataset(IterableDataset):
    def __init__(self, files):
        self.files = files

    def _get_sharded_lines(self, file_path):
        """確保多進程時，每個 worker 讀不同的部分"""
        worker_info = get_worker_info()
        if worker_info is None:  # 單進程模式
            start, step = 0, 1
        else:
            start, step = worker_info.id, worker_info.num_workers  # worker id & 總數

        with open(file_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i % step == start:  # 讓不同 worker 讀不同的行
                    yield line

    def __iter__(self):
        """讀取 JSONL 並轉換為合適格式"""
        sample_count = 0
        for file_path in self.files:
            for line in self._get_sharded_lines(file_path):
                raw_data = json.loads(line)

                # 把 list 轉回 torch.Tensor
                processed_data = {
                    key: torch.tensor(value) if isinstance(value, list) else value
                    for key, value in raw_data.items()
                }
                
                yield processed_data

            

train_files = ["processed_iterable_dataset/train_400k.jsonl"]
valid_files = ["processed_iterable_dataset/valid_50k.jsonl"] 
test_files = ["processed_iterable_dataset/test_50k.jsonl"]

train_dataset = ProcessedIterableDataset(train_files)

valid_dataset = ProcessedIterableDataset(valid_files)

test_dataset = ProcessedIterableDataset(test_files)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size = batch_size, num_workers = 4, drop_last = False)
valid_loader = DataLoader(valid_dataset, batch_size = batch_size, num_workers = 4, drop_last = False)
test_loader = DataLoader(test_dataset, batch_size = batch_size, num_workers = 4, drop_last = False)

In [10]:
import os

def count_lines(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

# 計算訓練資料大約的 batch 數
train_total_samples = sum(count_lines(f) for f in train_files)
train_approx_batches = train_total_samples // batch_size

valid_total_samples = sum(count_lines(f) for f in valid_files)
valid_approx_batches = valid_total_samples // batch_size

test_total_samples = sum(count_lines(f) for f in test_files)
test_approx_batches = test_total_samples // batch_size


print(f"Train 預計數量: {train_total_samples}, 預計 {train_approx_batches} 個 batch")
print(f"Valid 預計數量: {valid_total_samples}, 預計 {valid_approx_batches} 個 batch")
print(f"Test 預計數量: {test_total_samples}, 預計 {test_approx_batches} 個 batch")

Train 預計數量: 728821, 預計 45551 個 batch
Valid 預計數量: 91168, 預計 5698 個 batch
Test 預計數量: 90337, 預計 5646 個 batch


## Count

#### 計算 tag, scale, negative, time 的類別個數
計算後存成 JSON，之後可以直接用

In [11]:
# load counter result

def load_counter(target_attr):
    target_path = f'processed_iterable_dataset/counter/secbert_train_small_{target_attr}.json'

    with open(target_path, "r", encoding='utf-8') as f:
        data = json.load(f)
    return data

train_tag_counts = load_counter("tag")
train_tag_counts.pop('-100', None)
num_tag_samples = [train_tag_counts.get(str(tag2id[tag]), 0) for tag in tag_list]
print(f'Total tag class: {len(train_tag_counts)}')
train_time_counts = load_counter("time")
train_time_counts.pop('-100', None)
num_time_samples = [train_time_counts.get(str(time2id[time]), 0) for time in time_list]
print(train_time_counts)

train_neg_counts = load_counter("negative")
train_neg_counts.pop('-100', None)
num_neg_samples = [train_neg_counts.get(neg, 0) for neg in sorted(train_neg_counts.keys())]
print(train_neg_counts)

train_scale_counts = load_counter("scale")
train_scale_counts.pop('-100', None)
num_scale_samples = [train_scale_counts.get(str(scale2id[scale]), 0) for scale in scale_list]
print(train_scale_counts)

train_signed_scale_counts = load_counter("signed_scale")
train_signed_scale_counts.pop('-100', None)
num_signed_scale_samples = [train_signed_scale_counts.get(str(scale2id[scale]), 0) for scale in scale_list]
print(train_signed_scale_counts)

Total tag class: 978
{'6': 70691, '3': 175155, '1': 195615, '4': 124746, '0': 135307, '5': 10629, '2': 14182, '8': 490, '7': 351}
{'0': 701747, '1': 19527}
{'18': 328811, '10': 117673, '12': 152722, '15': 50895, '21': 18758, '8': 1695, '17': 38, '14': 13, '9': 20, '11': 18, '13': 17, '24': 30, '16': 37, '20': 2, '6': 5, '19': 3}
{'20': 109375, '19': 145929, '11': 55933, '18': 111058, '21': 48677, '12': 63773, '17': 55430, '44': 6455, '13': 29733, '15': 7704, '16': 17977, '9': 5417, '10': 27284, '42': 1885, '14': 12252, '22': 7546, '8': 2140, '46': 962, '43': 4839, '35': 285, '23': 780, '45': 3697, '24': 439, '36': 517, '37': 216, '38': 37, '40': 76, '41': 331, '7': 238, '34': 116, '47': 70, '6': 33, '33': 14, '39': 21, '48': 6, '5': 8, '4': 1}


## loss



In [12]:
# hits@k
import torch

def hits_at_k(predictions, targets, k=5):
    """
    計算 Hits@K 指標
    :param predictions: (batch_size, num_tags) - 預測分數
    :param targets: (batch_size, num_tags) - 目標標籤 (one-hot 或 multi-hot)
    :param k: 取前 K 個預測標籤
    :return: Hits@K 平均值
    """
    valid_mask = targets != -100  # 只保留有效的索引
    targets = targets[valid_mask]
    predictions = predictions[valid_mask]
    # print(f"targets.shape: {targets.shape}, targets min: {targets.min()}, targets max: {targets.max()}")
    # assert targets.min() >= 0, f"targets 包含負數: {targets}"

    top_k_preds = torch.topk(predictions, k, dim=-1).indices  # 取得 top-K 標籤索引

       # 確保 targets 維度正確
    if targets.dim() == 1:  # 若 targets 是索引格式 (batch_size,)
        targets = torch.nn.functional.one_hot(targets, num_classes=predictions.size(1))

    targets = targets.float()  # 確保是 float tensor

    # 判斷是否命中 top-K (batch_size, k) → (batch_size,)
    hits = torch.any(targets.gather(1, top_k_preds), dim=1).float()

    return hits.mean().item()  # 計算平均命中率

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
# device = 'cpu'

cuda


In [14]:
# time loss
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np
from torch import nn
# 取得 time 類別的出現次數
time_class_counts = torch.tensor(num_time_samples)

total_samples = sum(train_time_counts.values())
num_classes = len(train_time_counts)

time_class_weights = {
    cls: total_samples / (num_classes * count) 
    for cls, count in train_time_counts.items()
}
time_class_weights = dict(sorted(time_class_weights.items(), key = lambda item:item[0]))
time_class_weights = list(time_class_weights.values())

time_smoothed_weights = np.log1p(time_class_weights)

MIN_WEIGHT =  1 # 設定最小值
time_smoothed_weights = np.clip(time_smoothed_weights, MIN_WEIGHT, None)
time_smoothed_weights[0] = 1.5
time_smoothed_weights[6] = 1.5
print(time_smoothed_weights)
time_class_weights = torch.tensor(time_smoothed_weights, dtype=torch.float).to(device)
time_loss_fn = nn.CrossEntropyLoss(weight = time_class_weights, ignore_index = CLASSIFICATION_MISSING_VALUE)

[1.5        1.         1.90167407 1.         1.         2.15193528
 1.5        5.44323412 5.11132642]


In [15]:
# scale loss
scale_class_weights = {
    15: 1.2,  21: 1.5
}
num_classes = len(scale_list)
weights_list = [scale_class_weights.get(i, 1.0) for i in range(num_classes)]

scale_class_weights = torch.tensor(weights_list, dtype=torch.float).to(device)

# 定義 loss function
scale_loss_fn = nn.CrossEntropyLoss(weight=scale_class_weights, ignore_index=CLASSIFICATION_MISSING_VALUE)

In [16]:
# netagive loss

import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    '''https://doi.org/10.1109/tpami.2018.2858826'''
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean", ignore_index = CLASSIFICATION_MISSING_VALUE):
        """
        alpha: 平衡因子 (適用於正負類不平衡)
        gamma: 縮放因子 (讓難分類的樣本 loss 權重變高)
        reduction: 可選 ["mean", "sum", "none"]，控制 loss 的輸出方式
        """
        super(FocalLoss, self).__init__()
        if isinstance(alpha, (float, int)):  # 確保 alpha 是 tensor
            self.alpha = torch.tensor([1 - alpha, alpha])  # alpha_neg, alpha_pos
        else:
            self.alpha = torch.tensor(alpha) 
        self.gamma = gamma
        self.reduction = reduction
        self.ce_loss = nn.CrossEntropyLoss(reduction="none", ignore_index=CLASSIFICATION_MISSING_VALUE)
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        """
        logits: 預測值 (模型輸出，形狀 [batch_size, 2]，未經 softmax)
        targets: 標籤值 (形狀 [batch_size]，0 或 1)
        """
        device = logits.device
        self.alpha = self.alpha.to(device)
        # print('focal loss', targets)
        # print(targets.shape)
        if targets.dim() > 1:
            targets = targets.argmax(dim=-1)
        
         # 1. 移除 ignore_index
        valid_mask = (targets != self.ignore_index)
        targets = targets[valid_mask]
        logits = logits[valid_mask]

        # if targets.numel() == 0:  # 避免 loss 計算時出現空值
        #     return torch.tensor(0.0, device=device, requires_grad=True)
        # if torch.any(filtered_targets < 0) or torch.any(filtered_targets >= logits.shape[-1]):
        #     raise ValueError(f"Invalid target values detected: {filtered_targets}")
        
        # 2. 計算 CrossEntropyLoss
        ce_loss = self.ce_loss(logits, targets)  # 計算 cross entropy loss
        pt = torch.exp(-ce_loss)  # 選擇正確類別的機率
        
        # 4. 計算 focal loss 權重
        focal_weight = (1 - pt) ** self.gamma  # (1 - p_t)^gamma
        alpha_weight = self.alpha.gather(0, targets.data.view(-1))  # 根據 targets 索引 alpha

        loss = alpha_weight * focal_weight * ce_loss

        # 5. 根據 reduction 返回 loss
        if self.reduction == "mean":
            return loss.mean() if loss.numel() > 0 else torch.tensor(0.0, device=device, requires_grad=True)
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss  # 不做平均，返回 batch loss

neg_loss = FocalLoss(alpha=0.25, gamma=3.0, reduction="mean")

In [17]:
# tag loss
import torch
import torch.nn as nn
import torch.nn.functional as F

class CB_CE_Loss(nn.Module):
    '''
    https://ieeexplore.ieee.org/abstract/document/8953804
    '''
    def __init__(self, num_samples, beta=0.99, ignore_index=CLASSIFICATION_MISSING_VALUE):
        """
        Args:
            num_samples: list or tensor, 每個類別的樣本數
            beta: 控制 class-balanced 權重的超參數 (通常取 0.99)
        """
        super(CB_CE_Loss, self).__init__()
        
        # 計算 Class-Balanced 權重
        effective_num = 1.0 - torch.pow(torch.tensor(beta), torch.tensor(num_samples))
        weights = (1.0 - beta) / (effective_num + 1e-8)
        # self.weights = weights / torch.sum(weights)  # normalize
        self.weights = weights
        self.ignore_index = ignore_index
        
    def forward(self, logits, targets):
        """
        Args:
            logits: (batch_size, num_classes) 模型輸出的 logits
            targets: (batch_size,) 類別索引標籤
        Returns:
            CB-CE Loss 值
        """
        
        if targets.dim() > 1:
            targets = targets.argmax(dim=-1)
            
        valid_mask = (targets != self.ignore_index)  # 只對有效的 targets 計算 loss
        targets = targets[valid_mask]
        logits = logits[valid_mask]
        
        # 計算標準 CE Loss
        ce_loss = F.cross_entropy(logits, targets, reduction='none', ignore_index=self.ignore_index)
        
        # 依照類別權重調整 loss
        class_weights = self.weights.to(logits.device)
        weighted_loss = ce_loss * class_weights[targets]
        
        # weighted_loss = ce_loss * class_weights[targets] * weight_mask.float()
        return torch.mean(weighted_loss)

        # return weighted_loss.sum() / weight_mask.sum()  # 只對有效樣本取平均

# train_tag_counts = get_value_counts(train_loader, "tag")
# num_tag_samples = [train_tag_counts.get(tag, 0) for tag in sorted(train_tag_counts.keys())]  # 確保對應到索引順序
tag_loss_fn = CB_CE_Loss(num_tag_samples, beta = 0.99)
signed_scale_loss_fn  = CB_CE_Loss(num_signed_scale_samples, beta = 0.99)

In [18]:
# train_time_counts = get_value_counts(train_loader, "time")
# num_time_samples = [train_time_counts.get(time, 0) for time in sorted(train_time_counts.keys())]
time_loss_fn = CB_CE_Loss(num_time_samples)

In [19]:
# fact loss

def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small_error = torch.abs(error) < delta
    squared_loss = 0.5 * error ** 2
    linear_loss = delta * (torch.abs(error) - 0.5 * delta)
    return torch.where(is_small_error, squared_loss, linear_loss).mean()

def signed_log(x):
    return torch.sign(x) * torch.log1p(torch.abs(x))  # 保留正負號

def fact_loss_fn(fact_pred, fact_target):
    # 過濾特殊值
    valid_mask = fact_target != NUMERIC_MISSING_VALUE
    fact_pred = fact_pred[valid_mask]
    fact_target = fact_target[valid_mask]
    
    #  signed log 轉換
    fact_target_log = signed_log(fact_target)
    fact_pred_log = signed_log(fact_pred)

    #  # 設定 Hybrid Loss 的閾值
    threshold = 8.0

    # # 小於 threshold 用 Huber Loss，大於 threshold 用 MSE
    use_huber = fact_target_log.abs() < threshold
    use_mse = ~use_huber

    huber_part = huber_loss(fact_pred_log[use_huber], fact_target_log[use_huber]) if use_huber.any() else 0
    mse_part = mse_loss(fact_pred_log[use_mse], fact_target_log[use_mse]) if use_mse.any() else 0

    # 最終 loss
    return huber_part + mse_part
    

In [20]:
# loss function

classification_loss = nn.CrossEntropyLoss(ignore_index = CLASSIFICATION_MISSING_VALUE)
mse_loss = nn.MSELoss()

def compute_loss(outputs, targets, values, task_weights = None, hits_k = False):
    losses = {}
    tag_hits_k = {}
    if "tag" in targets:
        losses["tag"] = tag_loss_fn(outputs["tag"], targets["tag"])

        # if not model.training:
        if hits_k:
            tag_hits_k["hits_1"] = hits_at_k(outputs["tag"], targets["tag"], k = 1)
            tag_hits_k["hits_3"] = hits_at_k(outputs["tag"], targets["tag"], k = 3)
            tag_hits_k["hits_5"] = hits_at_k(outputs["tag"], targets["tag"], k = 5)

    if "time" in targets:
        losses["time"] = time_loss_fn(outputs["time"], targets["time"])
    
    if "scale" in targets:
        losses["scale"] = scale_loss_fn(outputs["scale"], targets["scale"])
    
    if "negative" in targets:
        # 負號預測
        negative_pred = outputs["negative"].argmax(dim=-1)  # [batch_size]
        losses["negative"] = neg_loss(outputs["negative"], targets["negative"])
    # --- NEW: Signed Scale Loss ---
    if "signed_scale" in targets and "signed_scale" in outputs:
        # This matches the logic for your new SignedScaleModel
        losses["signed_scale"] = signed_scale_loss_fn(outputs["signed_scale"], targets["signed_scale"])
  
    #     
    total_loss = sum(task_weights[k] * losses[k] for k in losses.keys())
    
    # 平衡 loss 避免變大
    total_loss = total_loss / sum(task_weights.values())

    # print('total_loss', total_loss)
    return (total_loss, losses, tag_hits_k) if hits_k else (total_loss, losses)

## Model


In [21]:
# import torch
# import torch.nn as nn
# from transformers import BertModel

# class MultiTaskModel(nn.Module):
#     def __init__(self, bert_model_name, num_tags, num_times, num_scales):
#         super(MultiTaskModel, self).__init__()
#         self.bert = BertModel.from_pretrained(bert_model_name)
#         hidden_size = self.bert.config.hidden_size
        
#         # --- Existing Heads ---
#         self.tag_head = nn.Sequential(
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.LayerNorm(hidden_size // 2), 
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_size // 2, num_tags)
#         )

#         self.time_head = nn.Sequential(
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.LayerNorm(hidden_size // 2), 
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_size // 2, num_times)
#         )

#         # --- NEW: Combined Signed Scale Head ---
#         # The output dimension is num_scales * 2 
#         # (First half for positive scales, second half for negative scales)
#         self.num_signed_scales = num_scales * 2
        
#         self.signed_scale_head = nn.Sequential(
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.LayerNorm(hidden_size // 2), 
#             nn.GELU(),
#             nn.Dropout(0.5), # Kept the higher dropout from your original scale/negative heads
#             nn.Linear(hidden_size // 2, self.num_signed_scales) 
#         )
        
#         # Removed self.scale_head and self.negative_head
        
#     def forward(self, input_ids, attention_mask, start_tokens, end_tokens):
#         # BERT output
#         outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         sequence_output = outputs.last_hidden_state
        
#         # Aggregate target embeddings (Mean pooling over the span)
#         target_embeddings = [
#             sequence_output[i, start_tokens[i]:end_tokens[i] + 1].mean(dim=0)
#             for i in range(input_ids.size(0))
#         ] 
        
#         target_embeddings = torch.stack(target_embeddings) # (batch_size, hidden_size)
        
#         # Generate Logits
#         tag_logits = self.tag_head(target_embeddings)
#         time_logits = self.time_head(target_embeddings)
        
#         # --- NEW: Predict Signed Scale ---
#         signed_scale_logits = self.signed_scale_head(target_embeddings) 
        
#         return {
#             "tag": tag_logits,
#             "time": time_logits,
#             "signed_scale": signed_scale_logits, # Returns (batch_size, num_scales * 2)
#         }

In [22]:
import torch
import torch.nn as nn
from transformers import BertModel

class SignedScaleModel(nn.Module):
    def __init__(self, bert_model_name, num_scales):
        """
        Args:
            bert_model_name: Pretrained BERT model name (e.g., "nlpaueb/sec-bert-base")
            num_scales: The number of absolute scale classes (e.g., 25 for range -12 to 12).
                        The model will output num_scales * 2 classes.
        """
        super(SignedScaleModel, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size
        
        # Calculate total output classes: Positive(25) + Negative(25) = 50
        self.num_signed_scales = num_scales * 2
        
        # Single Head for Signed Scale
        self.signed_scale_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2), 
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_size // 2, self.num_signed_scales) 
        )
        
    def forward(self, input_ids, attention_mask, start_tokens, end_tokens):
        # 1. BERT Backbone
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        # 2. Span Aggregation (Mean Pooling)
        # Extracts embeddings for the specific token span identified by start/end_tokens
        target_embeddings = [
            sequence_output[i, start_tokens[i]:end_tokens[i] + 1].mean(dim=0)
            for i in range(input_ids.size(0))
        ]
        
        # Stack into (batch_size, hidden_size)
        target_embeddings = torch.stack(target_embeddings) 
        
        # 3. Prediction Head
        signed_scale_logits = self.signed_scale_head(target_embeddings) 
        
        # Return dictionary to match your training loop expectations
        return {
            "signed_scale": signed_scale_logits
        }

# train


In [42]:
# import wandb
# # 清 GPU
# with torch.no_grad():
#     torch.cuda.empty_cache()
# # torch.cuda.empty_cache()
# del model, input_ids, attention_mask, start_tokens, end_tokens, targets, outputs

# # 結束上次的紀錄
# wandb.finish()

### Train

#### Init & Setting

In [23]:
# import wandb
# # # 清 GPU
# # with torch.no_grad():
# #     torch.cuda.empty_cache()
# # # torch.cuda.empty_cache()
# # del model, input_ids, attention_mask, start_tokens, end_tokens, targets, outputs

# # # 結束上次的紀錄
# wandb.finish()

In [24]:
def get_task_weights(epoch):

    if epoch < 3:
       return {"tag": 15.0, "time": 1.0, "scale": 0.1, "negative": 3}  
    elif epoch < 8:
        return {"tag": 10.0, "time": 1.2, "scale": 0.2, "negative": 5}          
    elif epoch < 12:
        return {"tag": 10.0, "time": 1.5, "scale": 0.2, "negative": 5}  
    else:
        return {"tag": 8.0, "time": 1.0, "scale": 0.2, "negative": 3}


In [25]:
import wandb
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F
from transformers import get_scheduler
from tqdm import tqdm
import os

# --- Config ---
num_warmup_steps = 5
num_epochs = 30
eval_step = 20000 
# Since there is only one task now, the weight is technically irrelevant (it's just 1.0)
# but we keep the structure to avoid breaking your compute_loss function.
task_weights = {"signed_scale": 1.0} 

patience = 5
max_saved_models = 2
saved_models = [] 

model_name = "secbert-signed-scale" # Updated name
date = "0426"
index = 173
run_name = f"{model_name}-{date}-{index}"

# --- Model Initialization ---
# Assuming you are using the SignedScaleModel class defined in the previous step
model = SignedScaleModel(
    "nlpaueb/sec-bert-base",
    num_scales = len(scale_list) # e.g., 25
)

# --- Optimizer & Scheduler ---
bert_lr = 1e-5
signed_scale_head_lr = 3e-5 # Set a specific LR for the new combined head

optimizer = AdamW([
    # Group 1: BERT backbone
    {"params": model.bert.parameters(), "lr": bert_lr, "weight_decay": 1e-2}, 
    
    # Group 2: The new Signed Scale Head
    {"params": model.signed_scale_head.parameters(), "lr": signed_scale_head_lr,  "weight_decay": 1e-2}, 
])

# Note: Removed groups for tag_head, time_head, scale_head, negative_head

num_total_steps = num_epochs * train_approx_batches
scheduler = CosineAnnealingLR(optimizer, T_max = num_total_steps // 4, eta_min = 1e-7)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

best_val_loss = float("inf")
early_stop_counter = 0 

os.makedirs(f'model_weight/{model_name}', exist_ok=True)
os.makedirs(f'check_point/{model_name}', exist_ok=True)

# --- WandB Init ---
wandb.init(
    project = "multi-task-model", # You might want to rename this project to "signed-scale-model"
    resume="allow",
    id='wqugd5cx', # Ensure this ID is correct for resuming, or remove if starting fresh
    name = f'{run_name}_cont_0330_173',
    config = {
        "learning_rates": { 
            "bert": bert_lr,
            "signed_scale_head": signed_scale_head_lr, # Updated config
        },
        "epochs": num_epochs,
        "task_weights": task_weights,
        "train_data_size": train_total_samples,
        "valid_data_size": valid_total_samples,
        "eval_step": eval_step,
        "model": "SignedScaleModel"
    },
)

cuda


wandb: Currently logged in as: mo11om (mo1om) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


#### Validation

In [26]:
def validate_model(model, val_loader, task_weights, device):
    model.eval()
    val_loss = 0
    valid_batch_count = 0
    
    # 1. Update to track only the new task
    all_losses = {'signed_scale': 0}
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            start_tokens = batch["start_token"].to(device)
            end_tokens = batch["end_token"].to(device)
            values = batch["value"].to(device)
            
            # 2. Fetch the new target from the batch
            targets = {
                "signed_scale": batch["signed_scale"].to(device)
            }
            
            outputs = model(input_ids, attention_mask, start_tokens, end_tokens)
            
            # Ensure your compute_loss function is also updated to handle "signed_scale"
            loss, losses = compute_loss(outputs, targets, values, task_weights)
            
            val_loss += loss.item()
            valid_batch_count += 1
            
            # 3. Aggregate specific task losses
            for key in all_losses:
                loss_value = losses.get(key, 0)
                # Ensure we are adding a float, not a tensor
                if isinstance(loss_value, torch.Tensor):
                    loss_value = loss_value.item()
                all_losses[key] += loss_value

    # Calculate average loss
    if valid_batch_count > 0:
        val_loss /= valid_batch_count
        for key in all_losses:
            all_losses[key] /= valid_batch_count
    
    return val_loss, all_losses

#### Training

In [27]:
def save_checkpoint(model, optimizer, scheduler, epoch, step, save_path="checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None
    }
    torch.save(checkpoint, save_path)
    print(f"Checkpoint saved at {save_path}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # 1. Load Model
    # Note: This will fail if you try to load an OLD MultiTaskModel checkpoint 
    # into the NEW SignedScaleModel because the keys (head names) don't match.
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 2. Load Optimizer
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # 3. Load Scheduler (Safe check)
    if scheduler and checkpoint.get('scheduler_state_dict'):
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    step = checkpoint['step']
    print(f"Checkpoint loaded: epoch {epoch}, step {step}")
    return epoch, step

In [ ]:
# 一般的 loss
model = model.to(device)
progress_bar = tqdm(range(num_total_steps), desc = "Training", dynamic_ncols = True)
step = 0

for epoch in range(num_epochs):
    if early_stop_counter >= patience:
        print("Early stopping triggered before starting a new epoch. Training stopped.")
        break  # 停止整個 training loop
        
    model.train()
    print(f"Task weight: {task_weights}")
    
    train_loss = 0
    train_samples = 0
    train_batch_count = 0
    train_losses = {}
    
    for batch in train_loader:
        if early_stop_counter >= patience:
            print("Early stopping triggered during training. Stopping current epoch.")
            break 
            
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_tokens = batch["start_token"].to(device)
        end_tokens = batch["end_token"].to(device)
        values = batch["value"].to(device)
        
        # --- UPDATED: Load only the signed_scale target ---
        targets = {
            "signed_scale": batch["signed_scale"].to(device)
        }

        outputs = model(input_ids, attention_mask, start_tokens, end_tokens)
        loss, losses = compute_loss(outputs, targets, values, task_weights)
        # print(loss, losses)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
       
        # 累計 loss
        train_loss += loss.item()
        for key, val in losses.items():
            train_losses[key] = train_losses.get(key, 0) + val
            
        train_samples += len(batch["input_ids"])
        train_batch_count += 1
        
        step += 1
        epoch_progress = step / train_approx_batches
        
        progress_bar.set_postfix(
            loss = loss.item(),
            epoch = f"{epoch_progress:.2f}"
        )
        progress_bar.update(1)

        # --- UPDATED: Log only relevant LRs ---
        # param_groups[0] is BERT, param_groups[1] is signed_scale_head
        wandb.log({
            "step": step, 
            "epoch": epoch_progress,
            "lr_bert": optimizer.param_groups[0]['lr'],
            "lr_signed_scale_head": optimizer.param_groups[1]['lr'], 
            "loss": loss.item()
        })        
    
        if step % eval_step == 0:
            # Save checkpoint (Make sure to use the UPDATED save_checkpoint function)
            save_checkpoint(model, optimizer, scheduler, epoch, step, f'check_point/{model_name}/{date}_{index}_step{step}.pth')
            
            avg_train_loss = train_loss / train_batch_count if train_batch_count > 0 else 0
            train_loss_dict = {f"train_loss_{key}": train_losses[key] / train_batch_count for key in train_losses} if train_batch_count > 0 else {}
            
            # Reset train counters
            train_loss = 0
            train_batch_count = 0
            train_losses = {}

            # Validation (Make sure to use the UPDATED validate_model function)
            val_loss, val_losses = validate_model(model, valid_loader, task_weights, device)
            val_loss_dict = {f"val_loss_{key}": val_losses[key] for key in val_losses}
            
            model.train()
            
            wandb.log({
                "step": step, 
                "epoch_progress": epoch_progress,
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                **train_loss_dict,
                **val_loss_dict
            })
    
            print(f"\nStep {step} Epoch {epoch_progress:.2f}: \nTrain Loss = {avg_train_loss:.4f} Validation Loss = {val_loss:.4f}")
            print(f"Validation Loss Breakdown: {val_losses}")
                
        
            # Save model if validation loss improves
            if val_loss < best_val_loss:
                print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. Saving model...")
                best_val_loss = val_loss
                early_stop_counter = 0
                
                model_save_path_epoch = f"model_weight/{model_name}/{date}_{index}_step{step+1}.pt"
                torch.save(model.state_dict(), model_save_path_epoch)
                saved_models.append(model_save_path_epoch)
                
                if len(saved_models) > max_saved_models:
                    oldest_model = saved_models.pop(0) 
                    if os.path.exists(oldest_model):
                        os.remove(oldest_model)
                        print(f"Removed old model: {oldest_model}")
                        
            else:
                early_stop_counter += 1
                print(f"No improvement. Early stop counter: {early_stop_counter}/{patience}")
        
            # Early stopping check
            if early_stop_counter >= patience:
                print("Early stopping triggered. Training stopped.")
                break

# 結束 wandb
wandb.finish()

Training:   0%|          | 0/1366530 [00:00<?, ?it/s]

Task weight: {'signed_scale': 1.0}


/home/mo1om/code/miniconda3/envs/selfmix/lib/python3.10/site-packages/transformers/models/bert/modeling_bert.py:440: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at ../aten/src/ATen/native/transformers/hip/sdp_utils.cpp:505.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Training:   1%|▏         | 20000/1366530 [1:43:59<116:08:26,  3.22it/s, epoch=0.44, loss=0.00415]  

Checkpoint saved at check_point/secbert-signed-scale/0426_173_step20000.pth

Step 20000 Epoch 0.44: 
Train Loss = 0.0086 Validation Loss = 0.0029
Validation Loss Breakdown: {'signed_scale': 0.0029284624205013787}
Validation loss improved from inf to 0.0029. Saving model...


Training:   3%|▎         | 40000/1366530 [3:37:11<114:28:12,  3.22it/s, epoch=0.88, loss=0.000102]   

...

Training:  10%|█         | 140000/1366530 [13:04:42<105:31:47,  3.23it/s, epoch=3.07, loss=0.11]    

Checkpoint saved at check_point/secbert-signed-scale/0426_173_step140000.pth

Step 140000 Epoch 3.07: 
Train Loss = -0.1244 Validation Loss = 0.0017
Validation Loss Breakdown: {'signed_scale': 0.0016912639380812801}
No improvement. Early stop counter: 4/5


Training:  12%|█▏        | 160000/1366530 [14:56:52<101:43:51,  3.29it/s, epoch=3.51, loss=0.00253]    

Checkpoint saved at check_point/secbert-signed-scale/0426_173_step160000.pth

Step 160000 Epoch 3.51: 
Train Loss = -0.0233 Validation Loss = 0.0008
Validation Loss Breakdown: {'signed_scale': 0.0007938446551953236}
Validation loss improved from 0.0011 to 0.0008. Saving model...
Removed old model: model_weight/secbert-signed-scale/0426_173_step40001.pt


Training:  12%|█▏        | 160183/1366530 [15:07:19<102:08:15,  3.28it/s, epoch=3.52, loss=0.00623]   

### 恢復訓練

In [ ]:
import os
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import wandb

# --- Configuration ---
last_step = 820000
o_date = '0426'
# Update to new model name to avoid overwriting old multi-task checkpoints
new_model_name = "secbert-signed-scale" 

# --- 1. Model Initialization ---
# Use the new class defined previously
model = SignedScaleModel(
    "nlpaueb/sec-bert-base",
    num_scales=len(scale_list) 
)
model.to(device)

# --- 2. Optimizer Setup ---
bert_lr = 1e-5
signed_scale_head_lr = 3e-5  # New LR for the combined head

optimizer = AdamW([
    {"params": model.bert.parameters(), "lr": bert_lr, "weight_decay": 1e-2},   
    {"params": model.signed_scale_head.parameters(), "lr": signed_scale_head_lr,  "weight_decay": 1e-2},   
])

# Recalculate steps
num_total_steps = num_epochs * train_approx_batches
scheduler = CosineAnnealingLR(optimizer, T_max = num_total_steps // 4, eta_min = 1e-7)

# --- 3. Checkpoint Loading Strategy ---
checkpoint_path = f"check_point/{model_name}/{o_date}_{index}_step{last_step}.pth"
print(f"Attempting to load: {checkpoint_path}")

if os.path.exists(checkpoint_path):
    # Logic to handle potential architecture mismatch if loading OLD model into NEW model
    try:
        # Try standard load first
        start_epoch, start_step = load_checkpoint(model, optimizer, scheduler, checkpoint_path, device)
    except RuntimeError as e:
        print(f"⚠️ Standard load failed (expected if switching architectures). Loading BERT backbone only...")
        # Partial Load: Load only matching keys (BERT) so you don't lose backbone progress
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model_dict = model.state_dict()
        # Filter out keys that don't match (e.g., old tag_head, scale_head)
        pretrained_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if k in model_dict and v.shape == model_dict[k].shape}
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)
        
        # Reset optimizer/scheduler because structure changed
        start_epoch = 0
        start_step = 0 
        print(f"✅ Backbone loaded. Restarting optimization from step 0 for new heads.")
else:
    print('Checkpoint step not exist. Starting fresh.')
    start_epoch = 0
    start_step = 0

# --- 4. Training Loop ---
progress_bar = tqdm(range(start_step, num_total_steps), desc="Training", dynamic_ncols=True)
step = start_step

for epoch in range(start_epoch, num_epochs):
    if early_stop_counter >= patience:
        print("Early stopping triggered before starting a new epoch. Training stopped.")
        break 
        
    model.train()
    # Only one task now, weight is 1.0
    task_weights = {"signed_scale": 1.0} 
    print(f"Task weight: {task_weights}")
    
    train_loss = 0
    train_samples = 0
    train_batch_count = 0
    train_losses = {}
    
    for batch in train_loader:
        if early_stop_counter >= patience:
            print("Early stopping triggered during training. Stopping current epoch.")
            break 
            
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_tokens = batch["start_token"].to(device)
        end_tokens = batch["end_token"].to(device)
        values = batch["value"].to(device)
        
        # --- UPDATED: Only fetch the signed_scale target ---
        targets = {
            "signed_scale": batch["signed_scale"].to(device)
        }

        outputs = model(input_ids, attention_mask, start_tokens, end_tokens)
        loss, losses = compute_loss(outputs, targets, values, task_weights)
       
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
       
        # Aggregate loss
        train_loss += loss.item()
        for key, val in losses.items():
            train_losses[key] = train_losses.get(key, 0) + val
            
        train_samples += len(batch["input_ids"])
        train_batch_count += 1
        
        step += 1
        epoch_progress = step / train_approx_batches
        
        progress_bar.set_postfix(
            loss = loss.item(),
            epoch = f"{epoch_progress:.2f}"
        )
        progress_bar.update(1)

        # --- UPDATED: Log specific LRs ---
        wandb.log({
            "step": step, 
            "epoch": epoch_progress,
            "lr_bert": optimizer.param_groups[0]['lr'],
            "lr_signed_scale_head": optimizer.param_groups[1]['lr'],
            "loss": loss.item()
        })            
    
        if step % eval_step == 0:
            # Save checkpoint
            save_checkpoint(model, optimizer, scheduler, epoch, step, f'check_point/{new_model_name}/{date}_{index}_step{step}.pth')
            
            avg_train_loss = train_loss / train_batch_count if train_batch_count > 0 else 0
            train_loss_dict = {f"train_loss_{key}": train_losses[key] / train_batch_count for key in train_losses} if train_batch_count > 0 else {}
            
            train_loss = 0
            train_batch_count = 0
            train_losses = {}

            # Validation
            val_loss, val_losses = validate_model(model, valid_loader, task_weights, device)
            val_loss_dict = {f"val_loss_{key}": val_losses[key] for key in val_losses}
            
            model.train()
            
            wandb.log({
                "step": step, 
                "epoch_progress": epoch_progress,
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                **train_loss_dict,
                **val_loss_dict
            })
    
            print(f"\nStep {step} Epoch {epoch_progress:.2f}: \nTrain Loss = {avg_train_loss:.4f} Validation Loss = {val_loss:.4f}")
            print(f"Validation Loss Breakdown: {val_losses}")
                
            # Save model if validation loss improves
            if val_loss < best_val_loss:
                print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. Saving model...")
                best_val_loss = val_loss
                early_stop_counter = 0
                
                model_save_path_epoch = f"model_weight/{new_model_name}/{date}_{index}_step{step+1}.pt"
                torch.save(model.state_dict(), model_save_path_epoch)
                saved_models.append(model_save_path_epoch)
                
                if len(saved_models) > max_saved_models:
                    oldest_model = saved_models.pop(0) 
                    if os.path.exists(oldest_model):
                        os.remove(oldest_model)
                        print(f"Removed old model: {oldest_model}")
                        
            else:
                early_stop_counter += 1
                print(f"No improvement. Early stop counter: {early_stop_counter}/{patience}")
        
            if early_stop_counter >= patience:
                print("Early stopping triggered. Training stopped.")
                break

wandb.finish()